<a href="https://colab.research.google.com/github/rithikakknair/OIBSIP/blob/main/Python-Task3-Data-cleaning/Task3_Data_cleaning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np

# 1. Upload your messy dataset file
from google.colab import files
uploaded = files.upload()

# 2. Read the uploaded file automatically
filename = list(uploaded.keys())[0]
df = pd.read_csv(filename)

print("=== INITIAL DATA QUALITY REPORT ===")
print(f"Total Rows: {df.shape[0]}")
print(f"Total Columns: {df.shape[1]}")
print(f"Duplicate Rows: {df.duplicated().sum()}\n")

# 3. Create a comprehensive quality report table (nulls, percentages, data types)
quality_report = pd.DataFrame({
    'Data Type': df.dtypes,
    'Null Count': df.isnull().sum(),
    'Null Percentage (%)': (df.isnull().sum() / len(df) * 100).round(2)
})

display(quality_report)

Saving messy_dataset.csv to messy_dataset.csv
=== INITIAL DATA QUALITY REPORT ===
Total Rows: 20
Total Columns: 11
Duplicate Rows: 0



,Data Type,Null Count,Null Percentage (%)
Rank,int64,0,0.0
Peak,object,11,55.0
All Time Peak,object,14,70.0
Actual gross,object,0,0.0
Adjusted gross (in 2022 dollars),object,0,0.0
Artist,object,0,0.0
Tour title,object,0,0.0
Year(s),object,0,0.0
Shows,int64,0,0.0
Average gross,object,0,0.0


In [ ]:
# 1. Store original row count for comparison later
initial_row_count = df.shape[0]

# 2. Duplicate Removal
duplicate_count = df.duplicated().sum()
df = df.drop_duplicates()
print(f"Removed {duplicate_count} duplicate rows.")

# 3. Missing Data Handling (Strategy: Fill numeric with median, text/categorical with 'Unknown')
for col in df.columns:
    if df[col].isnull().sum() > 0:
        if df[col].dtype in ['int64', 'float64']:
            median_val = df[col].median()
            df[col] = df[col].fillna(median_val)
            print(f"Handled missing values in numeric column '{col}' -> filled with median: {median_val}")
        else:
            df[col] = df[col].fillna('Unknown')
            print(f"Handled missing values in categorical column '{col}' -> filled with 'Unknown'")

# 4. Data Type Corrections & Standardisation (Example: strip whitespace from text columns)
for col in df.select_dtypes(include=['object']).columns:
    df[col] = df[col].astype(str).str.strip().str.title()

print("\nCleaning Complete for Cell 2!")
print(f"Current Row Count: {df.shape[0]}")
display(df.head())

Removed 0 duplicate rows.
Handled missing values in categorical column 'Peak' -> filled with 'Unknown'
Handled missing values in categorical column 'All Time Peak' -> filled with 'Unknown'

Cleaning Complete for Cell 2!
Current Row Count: 20


,Rank,Peak,All Time Peak,Actual gross,Adjusted gross (in 2022 dollars),Artist,Tour title,Year(s),Shows,Average gross,Ref.
0,1,1,2,"$780,000,000","$780,000,000",Taylor Swift,The Eras Tour †,2023–2024,56,"$13,928,571",[1]
1,2,1,7[2],"$579,800,000","$579,800,000",Beyoncé,Renaissance World Tour,2023,56,"$10,353,571",[3]
2,3,1[4],2[5],"$411,000,000","$560,622,615",Madonna,Sticky & Sweet Tour ‡[4][A],2008–2009,85,"$4,835,294",[6]
3,4,2[7],10[7],"$397,300,000","$454,751,555",Pink,Beautiful Trauma World Tour,2018–2019,156,"$2,546,795",[7]
4,5,2[4],Unknown,"$345,675,146","$402,844,849",Taylor Swift,Reputation Stadium Tour,2018,53,"$6,522,173",[8]


In [ ]:
# 1. Outlier Detection & Handling (Using IQR method to cap outliers in numeric columns)
for col in df.select_dtypes(include=['int64', 'float64']).columns:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    # Cap outliers to bounds
    df[col] = np.where(df[col] < lower_bound, lower_bound, df[col])
    df[col] = np.where(df[col] > upper_bound, upper_bound, df[col])
    print(f"Handled outliers in '{col}' using IQR Capping.")

# 2. Before vs. After Summary Table
summary_comparison = pd.DataFrame({
    'Metric': ['Total Rows', 'Duplicate Rows', 'Total Null Count'],
    'Before Cleaning': [initial_row_count, duplicate_count, quality_report['Null Count'].sum()],
    'After Cleaning': [df.shape[0], df.duplicated().sum(), df.isnull().sum().sum()]
})

print("\n=== BEFORE VS. AFTER SUMMARY TABLE ===")
display(summary_comparison)

# 3. Save the cleaned dataset to a new CSV file
cleaned_filename = 'Cleaned_Messy_Dataset.csv'
df.to_csv(cleaned_filename, index=False)
print(f"\nCleaned dataset successfully saved as '{cleaned_filename}'!")

Handled outliers in 'Rank' using IQR Capping.
Handled outliers in 'Shows' using IQR Capping.

=== BEFORE VS. AFTER SUMMARY TABLE ===


,Metric,Before Cleaning,After Cleaning
0,Total Rows,20,20
1,Duplicate Rows,0,0
2,Total Null Count,25,0



Cleaned dataset successfully saved as 'Cleaned_Messy_Dataset.csv'!
